In [3]:
#importing all the required libraries


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.optimize import minimize
import pandas_datareader.data as web

print("All libraries imported successfully")

All libraries imported successfully


We would like to load, inspect and clean the data if required. I have the data fromthe official US treasury bonds website.

In [4]:
df=pd.read_csv('../data/raw/par-yield-curve-rates-1990-2023.csv')
df.head()

print("Shape")
print(df.shape)

print("\nColumns")
print(df.columns)

print("\ninfo")
print(df.info())


Shape
(8507, 10)

Columns
Index(['date', '6 mo', '1 yr', '2 yr', '3 yr', '5 yr', '7 yr', '10 yr',
       '20 yr', '30 yr'],
      dtype='str')

info
<class 'pandas.DataFrame'>
RangeIndex: 8507 entries, 0 to 8506
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    8507 non-null   str    
 1   6 mo    8506 non-null   float64
 2   1 yr    8506 non-null   float64
 3   2 yr    8506 non-null   float64
 4   3 yr    8506 non-null   float64
 5   5 yr    8506 non-null   float64
 6   7 yr    8506 non-null   float64
 7   10 yr   8506 non-null   float64
 8   20 yr   7567 non-null   float64
 9   30 yr   7512 non-null   float64
dtypes: float64(9), str(1)
memory usage: 747.8 KB
None


In [5]:
#Some cleaning

df = df.rename(columns={
    "date": "Date",
    "6 mo": "6M",
    "1 yr": "1Y",
    "2 yr": "2Y",
    "3 yr": "3Y",
    "5 yr": "5Y",
    "7 yr": "7Y",
    "10 yr": "10Y",
    "20 yr": "20Y",
    "30 yr": "30Y"
})

df.head()

,Date,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
0,01-02-1990,7.89,7.81,7.87,7.90,7.87,7.98,7.94,NaN,8.00
1,01-03-1990,7.94,7.85,7.94,7.96,7.92,8.04,7.99,NaN,8.04
2,01-04-1990,7.90,7.82,7.92,7.93,7.91,8.02,7.98,NaN,8.04
3,01-05-1990,7.85,7.79,7.90,7.94,7.92,8.03,7.99,NaN,8.06
4,01-08-1990,7.88,7.81,7.90,7.95,7.92,8.05,8.02,NaN,8.09


In [6]:
df["Date"] = pd.to_datetime(
    df["Date"].astype(str).str.strip(),
    format="mixed",
    dayfirst=False
)

df["Date"].head()

0   1990-01-02
1   1990-01-03
2   1990-01-04
3   1990-01-05
4   1990-01-08
Name: Date, dtype: datetime64[us]

In [7]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 8507 entries, 0 to 8506
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    8507 non-null   datetime64[us]
 1   6M      8506 non-null   float64       
 2   1Y      8506 non-null   float64       
 3   2Y      8506 non-null   float64       
 4   3Y      8506 non-null   float64       
 5   5Y      8506 non-null   float64       
 6   7Y      8506 non-null   float64       
 7   10Y     8506 non-null   float64       
 8   20Y     7567 non-null   float64       
 9   30Y     7512 non-null   float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 664.7 KB


,Date,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
0,1990-01-02,7.89,7.81,7.87,7.90,7.87,7.98,7.94,NaN,8.00
1,1990-01-03,7.94,7.85,7.94,7.96,7.92,8.04,7.99,NaN,8.04
2,1990-01-04,7.90,7.82,7.92,7.93,7.91,8.02,7.98,NaN,8.04
3,1990-01-05,7.85,7.79,7.90,7.94,7.92,8.03,7.99,NaN,8.06
4,1990-01-08,7.88,7.81,7.90,7.95,7.92,8.05,8.02,NaN,8.09


In [8]:
df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Date,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
0,1990-01-02,7.89,7.81,7.87,7.90,7.87,7.98,7.94,NaN,8.00
1,1990-01-03,7.94,7.85,7.94,7.96,7.92,8.04,7.99,NaN,8.04
2,1990-01-04,7.90,7.82,7.92,7.93,7.91,8.02,7.98,NaN,8.04
3,1990-01-05,7.85,7.79,7.90,7.94,7.92,8.03,7.99,NaN,8.06
4,1990-01-08,7.88,7.81,7.90,7.95,7.92,8.05,8.02,NaN,8.09


In [9]:
df.tail()

,Date,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
8502,2023-12-22,5.31,4.82,4.31,4.04,3.87,3.92,3.90,4.21,4.05
8503,2023-12-26,5.28,4.83,4.26,4.05,3.89,3.91,3.89,4.20,4.04
8504,2023-12-27,5.26,4.79,4.20,3.97,3.78,3.81,3.79,4.10,3.95
8505,2023-12-28,5.28,4.82,4.26,4.02,3.83,3.84,3.84,4.14,3.98
8506,2023-12-29,5.26,4.79,4.23,4.01,3.84,3.88,3.88,4.20,4.03


In [10]:
df.isna().sum()

Date      0
6M        1
1Y        1
2Y        1
3Y        1
5Y        1
7Y        1
10Y       1
20Y     940
30Y     995
dtype: int64

In [11]:
df[df["6M"].isna()]

,Date,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
5199,2010-10-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.isna().sum()

Date      0
6M        1
1Y        1
2Y        1
3Y        1
5Y        1
7Y        1
10Y       1
20Y     940
30Y     995
dtype: int64

We notice that we have about 1000 missing data points for the 20 and 30 maturity bonds. for the sake of implementation and since its a prioject to learn. I have decided to drop these so that we have complete data on which PCA works really well. Future aspects of this project if extended could perhaps implemebt Nelson Siegel Svenson to interpolate and have an estimate of the missing maturity values.

In [16]:
# Converting the "Date" column to datetime format

df["Date"]=pd.to_datetime(df["Date"])

#Sorting by date in ascending order

df=df.sort_values("Date")

# Resetting the index to date

df=df.set_index("Date")

In [19]:
#checks
df.dtypes

df.head()

,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,
1990-01-02,7.89,7.81,7.87,7.90,7.87,7.98,7.94,NaN,8.00
1990-01-03,7.94,7.85,7.94,7.96,7.92,8.04,7.99,NaN,8.04
1990-01-04,7.90,7.82,7.92,7.93,7.91,8.02,7.98,NaN,8.04
1990-01-05,7.85,7.79,7.90,7.94,7.92,8.03,7.99,NaN,8.06
1990-01-08,7.88,7.81,7.90,7.95,7.92,8.05,8.02,NaN,8.09


In [23]:
#Now we drop the 20 the 30 year columns as I mentioned earlier, we will only use the 6M, 1Y, 2Y, 3Y, 5Y, 7Y and 10Y columns for our PCA analysis.

yield_curve=df[["6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]]

yield_curve.head()


,6M,1Y,2Y,3Y,5Y,7Y,10Y
Date,,,,,,,
1990-01-02,7.89,7.81,7.87,7.90,7.87,7.98,7.94
1990-01-03,7.94,7.85,7.94,7.96,7.92,8.04,7.99
1990-01-04,7.90,7.82,7.92,7.93,7.91,8.02,7.98
1990-01-05,7.85,7.79,7.90,7.94,7.92,8.03,7.99
1990-01-08,7.88,7.81,7.90,7.95,7.92,8.05,8.02


In [ ]:
#Now we check what row is actually missing from the data

yield_curve[yield_curve.isna().any(axis=1)]

#Now we drop the row with missing values

,6M,1Y,2Y,3Y,5Y,7Y,10Y
Date,,,,,,,
2010-10-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
yield_curve=yield_curve.dropna()

#and we check again if there are any missing values

yield_curve.isna().sum()

6M     0
1Y     0
2Y     0
3Y     0
5Y     0
7Y     0
10Y    0
dtype: int64

Now we calculate daily changes. This is the data that we will use for the PCA. We will also drop any rows that 0.

Why? We want to study how the rates move with eachother. Thats what PCA does. so that change is more important than preserving the longterm rates for the analysis. That is what the .diff() function does

In [ ]:
#this calculates the daily changes in the yield curve and drops any rows with missing values

yield_changes=yield_curve.diff().dropna()

In [30]:
yield_changes.head()

,6M,1Y,2Y,3Y,5Y,7Y,10Y
Date,,,,,,,
1990-01-03,0.05,0.04,0.07,0.06,0.05,0.06,0.05
1990-01-04,-0.04,-0.03,-0.02,-0.03,-0.01,-0.02,-0.01
1990-01-05,-0.05,-0.03,-0.02,0.01,0.01,0.01,0.01
1990-01-08,0.03,0.02,0.00,0.01,0.00,0.02,0.03
1990-01-09,-0.06,-0.03,0.01,-0.01,0.00,0.00,0.00


Now we do some sanity checks on the data

In [38]:

print(yield_changes.shape)

yield_changes.isna().sum()

yield_changes = yield_changes.round(6)

(8505, 7)


In [39]:
#Now we save the cleaned data to a new CSV file
yield_curve.to_csv('../data/processed/cleaned_yield_curve.csv')

yield_changes.to_csv('../data/processed/cleaned_yield_changes.csv')

This concludes the cleaning part of the data. We made some judgement calls for the sake of clarity and concept. The cleaned files are now available at the data/processed folder in csv format. We have also rounded to the 6th decimal place to ensure there is uniformity. thse changes are negligeble during the PCA.